# skinsa: Skin Lesion Classification on Imbalanced Data Using Deep Learning with Soft Attention
### Sensors 2022 Paper Replication & Enhancement on HAM10000

This notebook runs the full GPU training pipeline for:
1. **Baseline Protocol (`--protocol paper`):** 90/10 stratified split evaluated on validation set.
2. **Enhanced Protocol (`--protocol lesion`):** 80/10/10 `StratifiedGroupKFold(10)` grouped by `lesion_id` + **8-view Dihedral Test-Time Augmentation (TTA)**.
3. **Outputs:** Exports `results_bundle.zip` for instant upload into the web UI dashboard.

In [ ]:
# 1. GPU Check & Environment Setup
!nvidia-smi
!pip install "tensorflow>=2.16" "keras>=3.0" scikit-learn pandas numpy matplotlib pillow fastapi uvicorn

In [ ]:
# 2. Clone Repository or Upload skinsa Package
# If using Google Drive or git:
# !git clone <your-repo-url> skinsa_repo
# %cd skinsa_repo
# !pip install -e .
import tensorflow as tf, keras
print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)

In [ ]:
# 3. Download HAM10000 Dataset via Kaggle API
# Upload your kaggle.json before running:
# from google.colab import files
# files.upload() # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/ham10000 --unzip
!ls -lh data/ham10000

In [ ]:
# 4. Run Baseline Protocol: MobileNetV3Large (Paper Replication)
!python -m skinsa.cli --protocol paper --backbone mobilenet_v3_large --epochs 250 --batch-size 32

In [ ]:
# 5. Run Enhanced Protocol: MobileNetV3Large + 8-view Dihedral TTA
!python -m skinsa.cli --protocol lesion --backbone mobilenet_v3_large --epochs 250 --batch-size 32

In [ ]:
# 6. (Optional) Run DenseNet201 (Takes ~4x longer than MobileNetV3Large)
# !python -m skinsa.cli --protocol paper --backbone densenet201 --epochs 250 --batch-size 32
# !python -m skinsa.cli --protocol lesion --backbone densenet201 --epochs 250 --batch-size 32

In [ ]:
# 7. Bundle Results for Web UI Import
!zip -r results_bundle.zip outputs/
from google.colab import files
files.download('results_bundle.zip')